# Tech Challenge Fase 2
## 04.1 — Streaming Simulador de Eventos

Gera pequenos arquivos JSON para simular micro-batches de eventos educacionais.

## 1. Imports

In [0]:
import json, uuid, random, time
from datetime import datetime, timezone

## 2. Configuração

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"
config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))
INPUT_PATH = f"{config['paths']['streaming_path']}/input"

## 3. Parâmetros

In [0]:
municipios = [
    {"co_uf":"35","sg_uf":"SP","co_municipio":"3550308","no_municipio":"São Paulo"},
    {"co_uf":"33","sg_uf":"RJ","co_municipio":"3304557","no_municipio":"Rio de Janeiro"},
    {"co_uf":"31","sg_uf":"MG","co_municipio":"3106200","no_municipio":"Belo Horizonte"},
    {"co_uf":"29","sg_uf":"BA","co_municipio":"2927408","no_municipio":"Salvador"}
]
event_types = ["indicador_atualizado","meta_atualizada","resultado_corrigido"]
indicadores = ["taxa_alfabetizacao","meta_2030","taxa_participacao"]

## 4. Função geradora

In [0]:
def gerar_evento():
    m = random.choice(municipios)
    indicador = random.choice(indicadores)
    valor = round(random.uniform(80,100),2) if indicador=="meta_2030" else round(random.uniform(45,95),2)
    return {
        "event_id": str(uuid.uuid4()),
        "event_timestamp": datetime.now(timezone.utc).isoformat(),
        "event_type": random.choice(event_types),
        "ano": 2025,
        "co_uf": m["co_uf"],
        "sg_uf": m["sg_uf"],
        "co_municipio": m["co_municipio"],
        "no_municipio": m["no_municipio"],
        "indicador": indicador,
        "valor": valor,
        "origem": "simulador_fiap",
        "event_version": 1
    }

## 5. Geração dos micro-batches

In [0]:
for batch in range(1,4):
    eventos = [gerar_evento() for _ in range(10)]
    nome = f"eventos_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}_batch_{batch}.json"
    conteudo = "\n".join(json.dumps(e, ensure_ascii=False) for e in eventos)
    dbutils.fs.put(f"{INPUT_PATH}/{nome}", conteudo, overwrite=False)
    print(f"Batch {batch}: {len(eventos)} eventos")
    time.sleep(1)